In [1]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
import torchvision.models as models
from torch import nn

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

In [2]:
print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [3]:
for params in model.parameters():
    params.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features=1280, out_features=2)
)

In [4]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

# Keep eval transform fixed (same one used by camera.py) and add light train augmentations
train_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(train_transforms)
print(test_transforms)

Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    RandomCrop(size=(224, 224), padding=None)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=None, hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [5]:
# Data part comes here
from torchvision import datasets
from pathlib import Path

train_dir = Path("data/train")
test_dir = Path("data/test")

train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
test_data = datasets.ImageFolder(test_dir, transform=test_transforms)

num_workers = 2
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    dataset=train_data,
    shuffle=True,
    batch_size=32,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=(num_workers > 0)
 )

test_loader = DataLoader(
    dataset=test_data,
    shuffle=False,
    batch_size=32,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=(num_workers > 0)
 )

print(f"Classes: {train_data.classes}")
print(f"Train: {len(train_data)} | Test: {len(test_data)}")
print(f"num_workers={num_workers} | pin_memory={pin_memory}")

Classes: ['Drowsy', 'Non Drowsy']
Train: 33434 | Test: 8359
num_workers=2 | pin_memory=True


In [6]:
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

for p in model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True
for p in model.features[-2:].parameters():
    p.requires_grad = True

params_to_train = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    params_to_train,
    lr=3e-4,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
            pred = model(X)
            loss = loss_fn(pred, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        correct += (pred.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def test_epoch(model, loader):
    model.eval()
    correct = 0
    with torch.inference_mode():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            correct += (pred.argmax(dim=1) == y).sum().item()
    return correct / len(loader.dataset)

best_test_acc = 0.0
best_state_dict = copy.deepcopy(model.state_dict())

for epoch in range(15):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn)
    test_acc = test_epoch(model, test_loader)
    scheduler.step(test_acc)
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_state_dict = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch+1} | Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | Test Acc: {test_acc:.2f} | Best: {best_test_acc:.2f}")

model.load_state_dict(best_state_dict)
print(f"Loaded best checkpoint with Test Acc: {best_test_acc:.4f}")

Using device: cuda
Epoch 1 | Loss: 0.17 | Train Acc: 0.98 | Test Acc: 1.00 | Best: 1.00
Epoch 2 | Loss: 0.13 | Train Acc: 1.00 | Test Acc: 1.00 | Best: 1.00
Epoch 3 | Loss: 0.13 | Train Acc: 1.00 | Test Acc: 1.00 | Best: 1.00
Epoch 4 | Loss: 0.12 | Train Acc: 1.00 | Test Acc: 1.00 | Best: 1.00


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

save_path = Path("models")
save_path.mkdir(parents=True, exist_ok=True)

model_path = save_path / "drowsy_or_not.pth"
torch.save(model.state_dict(), model_path)
print(f"Best model saved at: {model_path.resolve()}")

Model saved at: C:\Users\Lenovo\Documents\AIML path\Driver attention state classifier\models\drowsy_or_not.pth


In [7]:
# Minimal robustness check on test set: confusion matrix + per-class precision/recall
model.eval()
all_preds, all_labels = [], []

with torch.inference_mode():
    for X, y in test_loader:
        X = X.to(device)
        logits = model(X)
        preds = torch.argmax(logits, dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(y)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
num_classes = len(train_data.classes)
cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)

for t, p in zip(all_labels, all_preds):
    cm[t.long(), p.long()] += 1

print("Confusion Matrix (rows=true, cols=pred):")
print(cm)

for i, class_name in enumerate(train_data.classes):
    tp = cm[i, i].item()
    fp = cm[:, i].sum().item() - tp
    fn = cm[i, :].sum().item() - tp
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    print(f"{class_name}: precision={precision:.4f}, recall={recall:.4f}")

overall_acc = (all_preds == all_labels).float().mean().item()
print(f"Overall test accuracy (recheck): {overall_acc:.4f}")

Confusion Matrix (rows=true, cols=pred):
tensor([[4469,    1],
        [   0, 3889]])
Drowsy: precision=1.0000, recall=0.9998
Non Drowsy: precision=0.9997, recall=1.0000
Overall test accuracy (recheck): 0.9999
